# Checkpoint follower iMD agent

Example of an iMD agent that guides selections to match checkpoints marked in an earlier recording.

<video src="../figures/imd-checkpoint-follower.webm" controls>

## Setup runner & utilities

In [1]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../systems/openmm/17-ala.openmm.zip")

OmniRunner.close_all_runners()
imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: imd checkpoint follower")
imd_runner.print_basic_info()
imd_runner.load(0)

Serving "EXAMPLE: imd checkpoint follower" (ws://localhost:50734), discoverable on all interfaces on port 54545
Available simulations:
[0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"
Switched to [0]: "17-ala.openmm"


In [2]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)
utilities.selections.update_selection("root", renderer="cartoon")

## Extract checkpoints from recording

Functions to extract checkpoints from a recording made using the "mobile restraints" example:

In [3]:
from dataclasses import dataclass, field

import numpy as np
import numpy.typing as npt
from MDAnalysis import Universe

from nanover.recording.reading import RecordingEvent
from nanover.imd.imd_state import dict_to_interaction

from iter_checkpoints import iter_checkpoints


@dataclass(kw_only=True, eq=False)
class TargetGroup:
    particles: list[int] = field(default_factory=list)
    position: npt.NDArray[np.float32]
    centroid: npt.NDArray[np.float32]


@dataclass(kw_only=True, eq=False)
class KeyFrame:
    targets: list[TargetGroup] = field(default_factory=list)
    centroids: npt.NDArray[np.float32] = field(init=False)

    def __post_init__(self):
        self.centroids = np.array([target.centroid for target in self.targets])


def extract_keyframes_restraints(path):
    keyframes = [restraint_keyframe_from_checkpoint(checkpoint) for checkpoint in iter_checkpoints(path)]
    keyframes = [keyframe for keyframe in keyframes if keyframe is not None]
    return keyframes


def restraint_keyframe_from_checkpoint(checkpoint: RecordingEvent):
    targets = []
    positions = checkpoint.next_frame.particle_positions

    # find active restraints at this checkpoint
    restraints = [
        dict_to_interaction(value)
        for key, value in checkpoint.next_state.items()
        if key.startswith("interaction.restraint")
    ]

    # keyframe is a target for each restraint
    for restraint in restraints:
        particles = [int(i) for i in restraint.particles]
        centroid = np.average(positions[particles], axis=0)

        target = TargetGroup(
            particles=particles,
            centroid=centroid,
            position=np.array(restraint.position),
        )
        targets.append(target)

    if not targets:
        return None

    targets.sort(key=lambda target: min(target.particles))
    return KeyFrame(targets=targets)

In [4]:
KEYFRAMES = extract_keyframes_restraints("../systems/recordings/knot-tying-checkpoints.nanover.zip")
print(len(KEYFRAMES))

8


## Old code

In [5]:
def residue_keyframe_from_checkpoint(checkpoint: RecordingEvent, universe: Universe):
    targets = []
    positions = checkpoint.next_frame.particle_positions

    for residue in universe.residues:
        particles = [int(i) for i in residue.atoms.indices]
        centroid = np.average(positions[particles], axis=0)

        target = TargetGroup(
            particles=particles,
            centroid=centroid,
            position=centroid,
        )
        targets.append(target)

    if not targets:
        return None

    return KeyFrame(targets=targets)


def extract_keyframes_residues(path, universe: Universe):
    return [residue_keyframe_from_checkpoint(checkpoint, universe) for checkpoint in iter_checkpoints(path)]

## iMD agent that follows a path

Define an iMD agent that maintains a single iMD interaction moving along a set path at a set constant speed:

In [6]:
from nanover.utilities.transforms import find_transformation_between_point_patterns, _transform_vec3s
from nanover.trajectory import FrameData


def fit_keyframe_to_frame(keyframe: KeyFrame, frame: FrameData):
    current = np.array([
        np.mean(frame.particle_positions[target.particles], axis=0)
        for target in keyframe.targets
    ])

    targets = _transform_vec3s(
        find_transformation_between_point_patterns(keyframe.centroids, current),
        keyframe.centroids,
    )

    return current, targets

In [7]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import ImdAgent, SceneObjectsUtility


class SmearAgent(ImdAgent):
    speed = 0.1
    keyframe: KeyFrame | None = None

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.objects = SceneObjectsUtility(self._app_server)

    def close(self, wait=False):
        super().close(wait=wait)
        self.objects.clear()

    def set_keyframe(self, keyframe: KeyFrame):
        self.interactions.clear()
        self.objects.clear()
        self.keyframe = keyframe

    def update_interactions(self, full_frame: FrameData, frame_update: FrameData):
        if self.keyframe is None:
            return

        # fit keyframe targets to actual positions
        prev_centroids, next_centroids = fit_keyframe_to_frame(
            self.keyframe, full_frame
        )

        # find necessary motions
        deltas = next_centroids - prev_centroids

        # cap motions by speed
        lengths = np.linalg.norm(deltas, axis=1).reshape(-1, 1)
        cappeds = deltas / lengths
        np.clip(lengths, max=self.speed, out=lengths)
        cappeds *= lengths

        # determine final interaction positions
        target_centroids = prev_centroids + cappeds

        with (
            self.interactions as interactions,
            self.objects as objects,
        ):
            points = [target_centroids[i] for i in range(len(self.keyframe.targets))]
            white = [1, 1, 1, .5]

            objects.update_line("backbone", positions=points, color=white, size=0.05)

            for i, target in enumerate(self.keyframe.targets):
                error = min(1.0, np.linalg.norm(deltas, axis=1)[i])
                label = f"error: {error:.2g}"
                color = [1, 1 - error, 1 - error, 1]
                position = points[i]

                objects.update_shape(i, shape="sphere", position=position, color=color)
                objects.update_label(i, text=label, position=position, color=color)

                if lengths[i] < 0.0001:
                    continue

                interaction = ParticleInteraction(
                    particles=target.particles,
                    position=list(target_centroids[i]),
                    interaction_type="spring",
                    scale=1000,
                    max_force=10000,
                )
                interactions.update_interaction(
                    f"interaction.REPLAYER.{i}", interaction
                )


## Controls

In [8]:
KEYFRAME_INDEX = -1
AGENT: ImdAgent | None = None


def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = KEYFRAME_INDEX + 1

    if KEYFRAME_INDEX == len(KEYFRAMES):
        smear_cancel()
        return

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_match(keyframe):
    global AGENT

    if AGENT is None:
        AGENT = SmearAgent.from_runner(imd_runner)
        AGENT.speed = .2 #.1 / 30

    # assign_atoms(keyframe)

    AGENT.set_keyframe(keyframe)
    AGENT.start()


def smear_cancel():
    global AGENT, KEYFRAME_INDEX

    if AGENT is None:
        return

    AGENT.close()
    AGENT = None
    KEYFRAME_INDEX = -1

    utilities.notify_all(f"RESETTING AGENT")


utilities.define_command("user/prev", label="prev checkpoint", handler=smear_back, icon="⏮️")
utilities.define_command("user/next", label="next checkpoint", handler=smear_next, icon="⏭️")
utilities.define_command("user/cancel", label="cancel", handler=smear_cancel, icon="⏹️")